In [ ]:
# -*- coding: utf-8 -*-
"""BigAlpha E2E raw 5m ridge sequence submission v4.

官方评测调用 main(datasources, start_date, end_date)，返回 date/instrument/score 三列。
本 notebook 只使用官方注入的分钟级原始量价与盘口字段；公榜阶段优先加载内嵌 v4 JSON 权重，
私榜或需要重训时可调用 train_and_save(datasources) 重新生成文本模型文件。
"""
import json
import os
import time
import numpy as np
import pandas as pd

MODEL_PATH = os.path.join(os.getcwd(), "bigalpha_ridge_model.json")
TRAIN_START = "2022-01-01"
TRAIN_END = "2023-12-31 23:59:59"
MAX_TRAIN_INSTRUMENTS = 200
EMBEDDED_MODEL_JSON = "{\"schema_version\":\"bigalpha_ridge_sequence_model_v1\",\"source_run_id\":\"real5m_fullraw_wf_v4_06_b5_raw_a0p005\",\"fields\":[\"open\",\"high\",\"low\",\"close\",\"volume\",\"amount\",\"adjust_factor\",\"deal_number\",\"ask_price1\",\"ask_price2\",\"ask_price3\",\"bid_price1\",\"bid_price2\",\"bid_price3\",\"ask_volume1\",\"ask_volume2\",\"ask_volume3\",\"bid_volume1\",\"bid_volume2\",\"bid_volume3\",\"ask_num_orders1\",\"ask_num_orders2\",\"ask_num_orders3\",\"bid_num_orders1\",\"bid_num_orders2\",\"bid_num_orders3\"],\"bars_per_day\":5,\"alpha\":0.005,\"feature_mode\":\"raw\",\"feature_mean\":[19.498521979985874,19.54561365260908,19.458775375268296,19.500782165832913,57434.21558255897,582189.4897244449,13.230434099397888,72.39751608291637,19.410744639028064,19.422921193710007,19.4342201572551,19.472621872766442,19.4579269120803,19.443860614724905,119811.78388134381,114998.30596854896,104223.49127948534,179382.39026090063,119521.65847033595,101787.38338098642,27.401554681915655,22.793316654753394,18.057898498927806,36.53602573266619,25.268834882058613,20.560328806290208,19.500669049321044,19.540229092208826,19.454417083631235,19.49470496783418,61862.14110078628,624950.2387980712,13.230434099397888,71.80511079342388,19.40347909220869,19.413776626161606,19.425493566833566,19.462871515368168,19.448377233738512,19.432972837741342,124173.81740528949,117816.37891350964,105170.15755897069,176370.32743030737,119276.74837383846,99329.92660829164,28.624303073624016,23.007558970693353,18.493727662616156,37.14746247319514,25.62612580414582,20.477805575411008,19.49444781987138,19.537490171551227,19.45037544674778,19.493628305932855,69813.04072551822,708878.2164147625,13.230434099397888,77.60861329521086,19.399933523945663,19.407413867047985,19.417629378127437,19.454271979985613,19.436867583988732,19.421968191565636,127086.03529306648,118701.01963902787,104741.38849177984,169702.09933881345,117995.62482130092,95766.52946747676,28.06040028591851,23.16452823445318,18.945818441744102,35.63481057898499,24.89933881343817,20.183863473909934,19.492956576125753,19.52903413152256,19.449020907791542,19.488636347391157,92869.50360972123,919715.8053747318,13.230434099397888,89.28808077197999,19.386539849892884,19.394265725518466,19.40482505361,19.446101322373305,19.43042101501086,19.414602215868577,123865.4353466762,121918.56395639743,106176.53920657613,153894.68806290207,113443.75092923518,91133.41524303073,23.48547176554682,22.11254467476769,18.77373123659757,30.460686204431738,22.663456040028592,19.122444603288063,19.488158148677773,19.518792530378843,19.45602734095795,19.485380092923705,180681.73575768407,1790936.8683936696,13.230434099397888,115.8150464617584,19.494766261615684,19.38280771979984,19.396930486061276,19.477620979270977,19.420354538956513,19.40630325232325,124582.3383666905,120867.30830950679,105041.84640814867,149331.48407791278,110756.42975339528,87879.89329878485,21.20950679056469,21.524553252323088,18.941833452466046,31.223534667619727,22.77546461758399,18.93363116511794],\"feature_scale\":[24.996310445775553,25.060197725579755,24.940558112789443,24.998256262298074,148050.59311347964,1167023.5854514043,205.52499165032668,85.96969709142034,25.038637670582535,25.04916408836014,25.05836846985675,24.98788212458525,24.98062090245022,24.972895152417767,1910364.3004064902,413844.9942873634,368774.5185700195,1812978.3757267152,428060.234266243,334948.0362172897,156.42708770586808,47.44140313694242,39.28809311157944,233.82756148312887,55.2114448205409,45.36455429576677,24.99831664459579,25.05405748102956,24.932826787493887,24.990056264161012,152450.66475280435,1265496.686869464,205.52499165032668,80.04553459065751,25.03074327049152,25.04060770607675,25.05081862189369,24.97957319396711,24.972249820615527,24.962469550621964,1910855.9810044162,409869.4390076255,366686.0998644595,1711554.4382260034,408893.8391608119,320260.3836877277,157.80869718160494,46.72757109768532,40.62686116007703,233.17512872632992,53.04710535045623,44.1104465466085,24.989139051495645,25.051029007724303,24.927351636016855,24.987633507197287,156400.3105383858,1447609.4263510124,205.52499165032668,86.8548391171342,25.029418567138713,25.039040139556963,25.047571313353064,24.97886393758186,24.972096336151996,24.965442303956074,1903822.788987522,398989.72943602625,359969.37718387955,1592883.0467288217,399031.6892294597,309633.9930729744,157.31099270142275,45.86956407358919,41.154678180303385,230.48282219234048,51.288805106250464,43.830626152647774,24.984926420695682,25.035154679144632,24.926255147034063,24.979528243389804,195499.98978197976,1620204.5833685144,205.52499165032668,98.06562594303419,25.02047888089763,25.02888970085202,25.03738962398289,24.974099737826155,24.965611278152092,24.959208447770603,1888836.061179117,390668.5203282519,339234.39553577534,1405140.5868510979,361014.58203534054,285287.16596637596,153.32496406869106,44.42820488665232,38.278774790273694,223.40887078126715,47.91827777607556,40.48093052798226,24.978482943635438,25.023010899073746,24.93475958603989,24.97519753024392,388162.22394905606,3432856.6324041598,205.52499165032668,157.68714738304212,24.988117595948797,25.024891775413767,25.033796644032,24.96400232580094,24.964785524320387,24.95581615660078,1870007.819462635,383077.471475365,328663.42025014164,1143667.3215012066,335974.39043141354,269511.3659531869,152.7377212859163,43.252364292209215,37.663681773425694,220.7518740577558,46.923472309985826,39.978059901895385],\"coef\":[0.027276942293273397,0.13563534313155112,1.0497641420900727,-0.29118134849354105,0.00023462971090170218,-0.0011033391794253728,-1.1982517686794292e-05,0.0015233188775043713,0.03947170949933399,-0.0843777952579128,0.07010571351498189,0.057828322981835614,0.05371994218492464,-0.08546488863849569,-0.004769321270170271,1.2164093056754035e-05,-0.0008085926118736808,0.0011833020162857026,0.0003531360853252885,0.0006808884749243808,0.00043590492669909396,0.001051869788480094,0.0013340510070949632,0.003207087799224678,0.00019167622040453598,-0.00037807036414458026,-0.27099282086069787,0.31889025394238707,-1.2014626540267224,-0.046539323263865645,0.00019232813809902805,-0.0016432758234136281,-1.1985502500716831e-05,0.0014718805089719985,0.0011044553010099035,-0.06430183932516081,0.03521759248688694,0.03140898226024827,-0.08680034598966056,0.06048748305591936,0.0042108239540112,-0.00017262674335836168,-0.000290886945260966,0.001215251577350574,-0.00014926193542870203,0.0005594788307319907,-0.0019612586112159355,-0.0013233248179389198,-0.0009180701045193459,-0.005461784436737221,-0.0009721900680558764,-0.00033041374007893617,0.5302959971881238,0.07449091788647647,-0.3082293746227674,-0.1742045458788135,-0.0007687673280332234,-0.0009950368068731164,-1.1986263140686555e-05,0.0033570627758896226,0.00277058268575268,0.029643644595363075,-0.05146242639136761,-0.04653038058459362,0.032350275009914234,0.0244042851634334,-0.002435708794451006,0.0010207026624677157,0.0002822917663908445,-0.0010878929034757057,0.001166943166702423,-0.00020272593607753938,-0.0024137669384011635,-0.0023418954326441263,-0.0007137679569803831,-0.0004282649487189834,-0.0007111727288777476,-0.000712404273185862,0.261360324692907,0.032786892047349914,-0.17768861602076344,0.47289979197099563,0.0008646114661178167,-0.0020015878035465704,-1.1987193613424317e-05,0.003825193392164201,0.01619518957166668,-0.017188366013854827,-0.0029839452956499273,0.007765914482943021,0.05179017034033341,-0.08833078551024134,0.006465340127145421,0.0007135256623144348,0.000894342413440083,-0.0023073132919696274,-0.0016961131607614076,-0.001336916738324152,0.0009733345803433437,0.0004973966776327357,0.0007917693806037746,0.00329126428129292,0.000607820894542248,0.0004646154019924155,0.15560605262497051,-0.03562148902746904,-0.32159775882170505,-0.3213570431017805,0.00018437396887887306,0.0030298860256067306,-1.1985033830986172e-05,-0.00646055244008808,-0.17222220385030743,0.022962534314223355,-0.014373946694700906,0.27196419759548596,-0.005322728489665259,0.0003150825769836999,-0.0031614771001174245,-0.0004213149869002812,-0.001597654001009828,0.0019065635649475365,0.0011391513139066304,0.001394933803471463,0.001211465920890605,0.0012620447998440403,0.0012248909536296743,0.0008463983977044636,-0.0001661252366511857,-0.0008923620010321464],\"intercept\":-0.0011307480012000802}"
MODEL_CONFIG = json.loads(EMBEDDED_MODEL_JSON)
FIELDS = tuple(MODEL_CONFIG["fields"])
BARS_PER_DAY = int(MODEL_CONFIG["bars_per_day"])
ALPHA = float(MODEL_CONFIG["alpha"])


def _logger():
    try:
        import structlog
        return structlog.get_logger()
    except Exception:
        class _Log:
            def info(self, *args, **kwargs):
                print(*args, kwargs if kwargs else "")
            def warning(self, *args, **kwargs):
                print(*args, kwargs if kwargs else "")
        return _Log()


def _pick_table(datasources):
    for key in ("bar5m", "bar_5m", "e2e_bar5m", "bar1m", "bar_1m"):
        if key in datasources:
            return datasources[key]
    if datasources:
        return next(iter(datasources.values()))
    raise ValueError("datasources is empty")


def _query_instrument_pool(start_date, end_date):
    import dai
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    if pool.empty:
        return pool
    pool = pool.copy()
    pool["date"] = pd.to_datetime(pool["date"], errors="coerce").dt.normalize()
    pool["instrument"] = pool["instrument"].astype(str)
    return pool.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])


def _available_columns(table, start_date, end_date):
    import dai
    try:
        sample = dai.query(f"SELECT * FROM {table} LIMIT 1", filters={"date": [start_date, end_date]}).df()
        return set(sample.columns)
    except Exception:
        return {"date", "instrument", *FIELDS}


def _query_raw_bars(table, start_date, end_date, instruments=None):
    import dai
    columns = _available_columns(table, start_date, end_date)
    available_fields = [field for field in FIELDS if field in columns]
    select_columns = ["date", "instrument", *available_fields]
    sql = f"SELECT {', '.join(select_columns)} FROM {table} ORDER BY instrument, date"
    filters = {"date": [start_date, end_date]}
    if instruments:
        filters["instrument"] = list(instruments)
    raw = dai.query(sql, filters=filters, compression=True).df()
    for field in FIELDS:
        if field not in raw.columns:
            raw[field] = np.nan
    return raw


def _build_feature_frame(raw):
    if raw.empty:
        return pd.DataFrame(columns=["date", "instrument"]), np.empty((0, len(FIELDS) * BARS_PER_DAY))
    work = raw.copy()
    work["timestamp"] = pd.to_datetime(work["date"], errors="coerce")
    work = work.dropna(subset=["timestamp", "instrument"])
    work["session_date"] = work["timestamp"].dt.strftime("%Y-%m-%d")
    work["instrument"] = work["instrument"].astype(str)
    for field in FIELDS:
        work[field] = pd.to_numeric(work[field], errors="coerce")

    vectors, rows, closes = [], [], []
    for (session_date, instrument), group in work.groupby(["session_date", "instrument"], sort=True):
        ordered = group.sort_values("timestamp").tail(BARS_PER_DAY)
        if ordered.empty:
            continue
        matrix = ordered.loc[:, FIELDS].to_numpy("float64")
        if len(matrix) < BARS_PER_DAY:
            pad = np.full((BARS_PER_DAY - len(matrix), len(FIELDS)), np.nan, dtype="float64")
            matrix = np.vstack([pad, matrix])
        else:
            matrix = matrix[-BARS_PER_DAY:]
        vectors.append(matrix.reshape(-1))
        rows.append({"date": session_date, "instrument": instrument})
        close_value = pd.to_numeric(ordered.get("close"), errors="coerce").iloc[-1] if "close" in ordered else np.nan
        closes.append(float(close_value) if np.isfinite(close_value) else np.nan)
    if not rows:
        return pd.DataFrame(columns=["date", "instrument"]), np.empty((0, len(FIELDS) * BARS_PER_DAY))
    keys = pd.DataFrame(rows)
    keys["close"] = closes
    return keys, np.vstack(vectors)


def _load_model(model_path=MODEL_PATH):
    if os.path.exists(model_path):
        with open(model_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
    else:
        payload = MODEL_CONFIG
    return {
        "fields": tuple(payload["fields"]),
        "bars_per_day": int(payload["bars_per_day"]),
        "alpha": float(payload.get("alpha", ALPHA)),
        "feature_mean": np.asarray(payload["feature_mean"], dtype="float64"),
        "feature_scale": np.asarray(payload["feature_scale"], dtype="float64"),
        "coef": np.asarray(payload["coef"], dtype="float64"),
        "intercept": float(payload["intercept"]),
    }


def _save_model(model, model_path=MODEL_PATH):
    payload = {
        "schema_version": "bigalpha_ridge_sequence_model_v1",
        "fields": list(model["fields"]),
        "bars_per_day": int(model["bars_per_day"]),
        "alpha": float(model["alpha"]),
        "feature_mode": "raw",
        "feature_mean": np.asarray(model["feature_mean"], dtype="float64").tolist(),
        "feature_scale": np.asarray(model["feature_scale"], dtype="float64").tolist(),
        "coef": np.asarray(model["coef"], dtype="float64").tolist(),
        "intercept": float(model["intercept"]),
    }
    with open(model_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, separators=(",", ":"))
    return model_path


def _predict(model, features):
    if features.size == 0:
        return np.asarray([], dtype="float64")
    mean = model["feature_mean"]
    scale = np.where(model["feature_scale"] > 1e-12, model["feature_scale"], 1.0)
    x = np.asarray(features, dtype="float64")
    x = np.where(np.isfinite(x), x, mean)
    return ((x - mean) / scale) @ model["coef"] + model["intercept"]


def _fit(features, labels):
    x = np.asarray(features, dtype="float64")
    mean = np.nanmean(x, axis=0)
    mean = np.where(np.isfinite(mean), mean, 0.0)
    x = np.where(np.isfinite(x), x, mean)
    scale = np.nanstd(x, axis=0)
    scale = np.where(scale > 1e-12, scale, 1.0)
    x = (x - mean) / scale
    y = np.asarray(labels, dtype="float64")
    y_mean = float(np.nanmean(y))
    coef = np.linalg.pinv(x.T @ x + ALPHA * np.eye(x.shape[1])) @ x.T @ (y - y_mean)
    return {
        "fields": FIELDS,
        "bars_per_day": BARS_PER_DAY,
        "alpha": ALPHA,
        "feature_mean": mean,
        "feature_scale": scale,
        "coef": coef,
        "intercept": y_mean,
    }


def _build_supervised(raw):
    keys, features = _build_feature_frame(raw)
    if keys.empty:
        raise RuntimeError("no usable daily sequence")
    labels = keys[["date", "instrument", "close"]].copy()
    labels["target"] = labels.groupby("instrument", sort=False)["close"].shift(-1) / labels["close"] - 1.0
    mask = np.isfinite(labels["target"].to_numpy("float64", na_value=np.nan))
    return features[mask], labels.loc[mask, "target"].to_numpy("float64")


def train_and_save(datasources, model_path=MODEL_PATH):
    logger = _logger()
    started = time.time()
    table = _pick_table(datasources)
    pool = _query_instrument_pool(TRAIN_START, TRAIN_END)
    instruments = pool["instrument"].drop_duplicates().tolist()[:MAX_TRAIN_INSTRUMENTS] if not pool.empty else None
    raw = _query_raw_bars(table, TRAIN_START, TRAIN_END, instruments)
    features, labels = _build_supervised(raw)
    model = _fit(features, labels)
    path = _save_model(model, model_path)
    logger.info("ridge model saved", path=path, samples=len(labels), elapsed=round(time.time() - started, 2))
    return path


def main(datasources, start_date, end_date):
    logger = _logger()
    started = time.time()
    table = _pick_table(datasources)
    start_ts = pd.to_datetime(start_date)
    query_start = (start_ts - pd.Timedelta(days=10)).strftime("%Y-%m-%d %H:%M:%S")
    pool = _query_instrument_pool(start_date, end_date)
    instruments = pool["instrument"].drop_duplicates().tolist() if not pool.empty else None
    raw = _query_raw_bars(table, query_start, end_date, instruments)
    keys, features = _build_feature_frame(raw)
    model = _load_model()
    pred = keys[["date", "instrument"]].copy()
    pred["score"] = _predict(model, features)
    pred["date"] = pd.to_datetime(pred["date"], errors="coerce").dt.normalize()
    pred["instrument"] = pred["instrument"].astype(str)
    pred = pred.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"], keep="last")

    if pool.empty:
        result = pred
    else:
        result = pool.merge(pred, how="left", on=["date", "instrument"])
        result["score"] = pd.to_numeric(result["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    logger.info("ridge prediction finished", rows=len(result), elapsed=round(time.time() - started, 2))
    return result[["date", "instrument", "score"]]


if __name__ == "__main__":
    train_and_save({"bar5m": "bigalpha_2026_stock_bar5m", "bar1m": "bigalpha_2026_stock_bar1m"})
